# 🧭 Human Activity Recognition Using the Forward & Backward Algorithms

**University Project | Statistical Machine Learning & Time Series Modeling**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dharshan007x/Human-Activity-Recognition---HMM-/blob/main/notebooks/HAR_HMM_Colab.ipynb)

**Problem Statement:** Identify human activities (*Walking*, *Sitting*, *Running*) from sequential 3D accelerometer readings. Model activities as hidden states and continuous sensor patterns as multivariate Gaussian emissions, then use the **Forward** and **Backward** algorithms to calculate sequence likelihoods and posterior state probabilities.

> **Academic Integrity & Implementation Note:**  
> All Forward, Backward, Posterior smoothing ($\gamma$), and Viterbi decoding routines in this project are implemented **manually from first principles in pure NumPy**. No third-party HMM libraries (`hmmlearn`, `pomegranate`, etc.) are used in the core code. `hmmlearn` is employed strictly as an independent cross-checking oracle within the unit test suite.

---
## 0. Environment Setup (Colab)
Clone the repository from GitHub and install all required dependencies.

In [ ]:
# Clone the repository
!git clone https://github.com/dharshan007x/Human-Activity-Recognition---HMM-.git /content/HAR_HMM

# Change working directory to the project root
import os
os.chdir('/content/HAR_HMM')
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install project dependencies
!pip install -q numpy>=1.24.0 scipy>=1.10.0 pandas>=2.0.0 scikit-learn>=1.2.0 plotly>=5.15.0 hmmlearn>=0.3.0 matplotlib
print("\n✅ All dependencies installed successfully.")

In [ ]:
# Verify project structure
import sys
sys.path.insert(0, '/content/HAR_HMM')

print("📂 Project structure:")
for root, dirs, files in os.walk('/content/HAR_HMM'):
    # Skip hidden dirs, __pycache__, .hypothesis, node_modules, frontend
    dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ('__pycache__', '.hypothesis', 'node_modules', 'frontend', 'backend', 'scratch')]
    level = root.replace('/content/HAR_HMM', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    if level < 2:
        subindent = '  ' * (level + 1)
        for file in files:
            print(f"{subindent}📄 {file}")

---
## 1. Mathematical Formulation

A discrete first-order Hidden Markov Model with continuous multivariate Gaussian emissions is defined by the tuple $\lambda = (\pi, A, B)$:

1. **Hidden State Space:** $S = \{s_1, s_2, s_3\}$ corresponding to `{0: Walking, 1: Sitting, 2: Running}`.
2. **Initial State Distribution:** $\pi_i = P(q_1 = s_i)$, where $\sum_{i=1}^N \pi_i = 1$.
3. **Transition Probability Matrix:** $A_{ij} = P(q_{t+1} = s_j \mid q_t = s_i)$, where $\sum_{j=1}^N A_{ij} = 1$.
4. **Continuous Emission Distribution:** $b_i(O_t) = \mathcal{N}(O_t; \mu_i, \Sigma_i)$ for feature vector $O_t \in \mathbb{R}^4$ ($[a_x, a_y, a_z, a_\text{mag}]$):
   $$\ln b_i(O_t) = -\frac{D}{2}\ln(2\pi) - \sum_{k=1}^D \ln L_{i,kk} - \frac{1}{2}\|v\|^2$$
   where $L_i L_i^T = \Sigma_i + \epsilon I$ (Cholesky factorization) and $L_i v = (O_t - \mu_i)$ (forward substitution).
5. **Forward Trellis Recursion:**
   $$\alpha_1(i) = \pi_i b_i(O_1)$$
   $$\alpha_{t+1}(j) = \left[\sum_{i=1}^N \alpha_t(i) A_{ij}\right] b_j(O_{t+1})$$
   $$P(O \mid \lambda) = \sum_{i=1}^N \alpha_T(i)$$
6. **Backward Trellis Recursion:**
   $$\beta_T(i) = 1$$
   $$\beta_t(i) = \sum_{j=1}^N A_{ij} b_j(O_{t+1}) \beta_{t+1}(j)$$
   $$P(O \mid \lambda) = \sum_{i=1}^N \pi_i b_i(O_1) \beta_1(i)$$
7. **Posterior Smoothing:**
   $$\gamma_t(i) = P(q_t = s_i \mid O, \lambda) = \frac{\alpha_t(i) \beta_t(i)}{\sum_{j=1}^N \alpha_t(j) \beta_t(j)}$$

In [ ]:
import numpy as np
import pandas as pd

from config import STATE_NAMES, NUM_STATES, RANDOM_SEED
from evaluation.metrics import compute_all_metrics
from models.backward import backward_pass
from models.forward import forward_pass
from models.hmm_model import GaussianHMM
from models.posterior import posterior_probabilities, predict_states
from models.viterbi import viterbi_decode
from preprocessing.clean import clean_accelerometer_data
from preprocessing.features import build_feature_matrix, segment_sequences
from preprocessing.filter import apply_butterworth_filter, downsample_signal
from preprocessing.split import SubjectScaler, subject_wise_split

np.random.seed(RANDOM_SEED)
print("✅ All core project modules successfully imported.")

---
## 2. Dataset Ingestion and Preprocessing Pipeline
We load the 100 Hz PAMAP2 accelerometer data (Chest IMU $\pm 16g$), apply physical bounds filtering ($[-16g, +16g]$), intra-subject linear interpolation for transient dropouts, and a 4th-order zero-phase Butterworth filter at 10 Hz before anti-aliasing decimation to 25 Hz ($M=4$).

In [ ]:
from preprocessing.loader import load_data

# Ingest PAMAP2 dataset (or validated physical synthetic fallback)
df_raw = load_data()
print(f"📊 Raw dataset shape: {df_raw.shape}")
print(f"👥 Subjects present: {sorted(df_raw['subject_id'].unique())}")
print(f"🏃 Activities present: {sorted(df_raw['activity_id'].unique())}")

# Clean physical bounds & interpolate within recording sessions
df_cleaned = clean_accelerometer_data(df_raw)

# Anti-aliasing low-pass filter and decimation: 100 Hz -> 25 Hz
df_filtered = apply_butterworth_filter(df_cleaned, cutoff_hz=10.0, fs=100.0, order=4)
df_downsampled = downsample_signal(df_filtered, original_fs=100, target_fs=25)
print(f"✅ Downsampled dataset shape: {df_downsampled.shape}")

---
## 3. Feature Construction and Subject-Wise Train/Test Split
To prevent any data leakage across participants, participants `101-106` form the training set, `107` the validation set, and `108-109` the strictly held-out test set. The `SubjectScaler` is fit **strictly on the training participants**.

In [ ]:
# Construct feature columns: [ax, ay, az, a_mag]
df_feats = build_feature_matrix(df_downsampled)

# Segment into contiguous sequences of length T=50 (2.0 seconds at 25 Hz)
sequences, seq_labels, seq_subjects = segment_sequences(df_feats, window_size=50, step_size=25)
print(f"🔢 Total segmented sequences: {len(sequences)}, sequence shape: {sequences[0].shape}")

# Subject-wise partition
splits = subject_wise_split(sequences, seq_labels, seq_subjects)
train_seqs, train_labels = splits["train"]
val_seqs, val_labels = splits["val"]
test_seqs, test_labels = splits["test"]

# Standardize features using training distribution only
scaler = SubjectScaler()
scaler.fit(train_seqs)
train_seqs_scaled = scaler.transform(train_seqs)
val_seqs_scaled = scaler.transform(val_seqs)
test_seqs_scaled = scaler.transform(test_seqs)

print(f"📐 Train: {len(train_seqs_scaled)}, Val: {len(val_seqs_scaled)}, Test: {len(test_seqs_scaled)} sequences")

---
## 4. Supervised HMM Parameter Estimation (Maximum Likelihood)
We compute empirical counts for initial state probabilities $\pi$ and transition matrix $A$, regularized with Laplace smoothing ($\alpha = 1.0$). Emission parameters $(\mu_i, \Sigma_i)$ are estimated per activity state with covariance jitter regularisation $\epsilon I = 10^{-4} I$.

In [ ]:
hmm = GaussianHMM(n_states=3, reg_covar=1e-4)
hmm.fit(train_seqs_scaled, train_labels, smoothing=1.0)

print("📊 Initial State Probabilities (π):")
for s_idx, name in enumerate(STATE_NAMES):
    print(f"  {name}: {hmm.pi[s_idx]:.4f}")

print("\n📊 Transition Probability Matrix (A):")
df_trans = pd.DataFrame(hmm.A, index=STATE_NAMES, columns=STATE_NAMES)
print(df_trans.round(4))

print("\n📊 Emission Means (μ):")
df_means = pd.DataFrame(hmm.means, index=STATE_NAMES, columns=["ax", "ay", "az", "a_mag"])
print(df_means.round(4))

---
## 5. Forward and Backward Inference Verification
We select an arbitrary held-out test sequence and run both the **Forward** and **Backward** algorithms in log space and scaled space.

**Crucial Theoretical Guarantee:**  
The total marginal observation sequence likelihood computed via Forward must equal the total likelihood computed via Backward:
$$\ln P(O \mid \lambda)_\text{Forward} = \ln \sum_{i=1}^N \alpha_T(i) \equiv \ln P(O \mid \lambda)_\text{Backward} = \ln \sum_{i=1}^N \pi_i b_i(O_1) \beta_1(i)$$

In [ ]:
# Select a test sequence
sample_seq = test_seqs_scaled[0]
sample_gt = test_labels[0]
print(f"🧪 Evaluating test sequence 0 (Ground Truth Activity: {sample_gt})")

# Compute emission log-likelihoods b_i(O_t)
from models.gaussian_emission import compute_log_emissions

log_B = compute_log_emissions(sample_seq, hmm.means, hmm.covars)

# 1. Forward Pass (Log-Space)
fwd_log_res = forward_pass(np.log(hmm.pi), np.log(hmm.A), log_B, use_scaling=False)
log_ll_fwd = fwd_log_res["log_likelihood"]

# 2. Backward Pass (Log-Space)
bwd_log_res = backward_pass(np.log(hmm.pi), np.log(hmm.A), log_B, use_scaling=False)
log_ll_bwd = bwd_log_res["log_likelihood"]

# 3. Scaled Normalization Pass (Rabiner formulation)
fwd_scaled_res = forward_pass(np.log(hmm.pi), np.log(hmm.A), log_B, use_scaling=True)
log_ll_scaled = fwd_scaled_res["log_likelihood"]

delta_fwd_bwd = abs(log_ll_fwd - log_ll_bwd)
delta_log_scaled = abs(log_ll_fwd - log_ll_scaled)

print(f"➡️  Forward Log-Likelihood:         {log_ll_fwd:.8f}")
print(f"⬅️  Backward Log-Likelihood:        {log_ll_bwd:.8f}")
print(f"📐 Scaled Forward Log-Likelihood:   {log_ll_scaled:.8f}")
print(f"📏 |Forward - Backward| Difference: {delta_fwd_bwd:.2e}")
print(f"📏 |Log - Scaled| Difference:       {delta_log_scaled:.2e}")

assert delta_fwd_bwd < 1e-8, f"Forward and Backward likelihoods diverge: {delta_fwd_bwd}"
assert delta_log_scaled < 1e-6, f"Log-space and Scaled likelihoods diverge: {delta_log_scaled}"
print("\n✅ [PASSED] Forward and Backward algorithms produce mathematically identical likelihoods.")

---
## 6. Posterior State Probability Smoothing
Using the Forward and Backward variables, we compute the smoothed state posteriors:
$$\gamma_t(i) = \frac{\alpha_t(i)\beta_t(i)}{P(O \mid \lambda)}$$
We verify that each time slice sums to 1.0 ($\sum_i \gamma_t(i) = 1.0$), and compare pointwise posterior argmax against global Viterbi joint-path decoding.

In [ ]:
gamma = posterior_probabilities(fwd_log_res["log_alpha"], bwd_log_res["log_beta"])

# Check row-stochasticity across all time steps
row_sums = gamma.sum(axis=1)
print(f"📐 Gamma shape (T × N): {gamma.shape}")
print(f"📏 Min row sum: {row_sums.min():.6f}, Max row sum: {row_sums.max():.6f}")
assert np.allclose(row_sums, 1.0, atol=1e-6)

# Pointwise decoding
posterior_preds = predict_states(gamma)
viterbi_path, vit_ll = viterbi_decode(np.log(hmm.pi), np.log(hmm.A), log_B)

print("\n🔍 Predicted state sequence (first 10 steps):")
print(f"  Posterior Argmax: {[STATE_NAMES[s] for s in posterior_preds[:10]]}")
print(f"  Viterbi Path:     {[STATE_NAMES[s] for s in viterbi_path[:10]]}")
print(f"  Path match rate:  {(posterior_preds == viterbi_path).mean() * 100:.1f}%")

---
## 7. Model Evaluation on Held-Out Test Subjects
We evaluate the trained HMM on the held-out test participant sequences (Subjects 108 & 109).

In [ ]:
y_pred = hmm.predict(test_seqs_scaled)
eval_metrics = compute_all_metrics(test_labels, y_pred, STATE_NAMES)

print(f"🏆 Test Set Evaluation Summary (Subjects 108 & 109, {len(test_labels)} sequences):")
print(f"  Accuracy:        {eval_metrics.accuracy:.4f}")
print(f"  Macro F1-Score:  {eval_metrics.macro_f1:.4f}")
print(f"  Weighted F1:     {eval_metrics.weighted_f1:.4f}")

print("\n📊 Per-Class Classification Report:")
df_class = pd.DataFrame(eval_metrics.per_class).T
print(df_class[["precision", "recall", "f1_score", "support"]].round(4))

print("\n📊 Normalized Confusion Matrix:")
df_cm = pd.DataFrame(eval_metrics.confusion_matrix_norm, index=STATE_NAMES, columns=STATE_NAMES)
print(df_cm.round(4))

---
## 8. Visualizations

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Forward Trellis Heatmap
log_alpha = fwd_log_res["log_alpha"]
alpha_norm = np.exp(log_alpha - log_alpha.max(axis=1, keepdims=True))

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    "Forward Trellis α(t, i) — Normalized",
    "Posterior Probabilities γ(t, i)"
), vertical_spacing=0.15)

fig.add_trace(go.Heatmap(
    z=alpha_norm.T, x=list(range(alpha_norm.shape[0])),
    y=STATE_NAMES, colorscale="YlOrRd",
    colorbar=dict(title="α (norm)", y=0.78, len=0.4)
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=gamma.T, x=list(range(gamma.shape[0])),
    y=STATE_NAMES, colorscale="Tealgrn",
    colorbar=dict(title="γ", y=0.22, len=0.4)
), row=2, col=1)

fig.update_layout(
    height=500, title_text="HMM Trellis & Posterior Visualization",
    template="plotly_white"
)
fig.update_xaxes(title_text="Time Step (t)", row=2, col=1)
fig.show()

In [ ]:
# Confusion Matrix Visualization
fig_cm = go.Figure(data=go.Heatmap(
    z=eval_metrics.confusion_matrix_norm,
    x=STATE_NAMES, y=STATE_NAMES,
    colorscale="YlOrRd",
    text=[[f"{val:.2%}" for val in row] for row in eval_metrics.confusion_matrix_norm],
    texttemplate="%{text}",
    textfont={"size": 16},
    hoverinfo="z"
))

fig_cm.update_layout(
    title="Normalized Confusion Matrix (Test Set)",
    xaxis_title="Predicted Activity",
    yaxis_title="True Activity",
    height=450, width=500,
    template="plotly_white"
)
fig_cm.show()

In [ ]:
# Sensor Signal Visualization for the test sample
fig_signal = make_subplots(rows=3, cols=1, shared_xaxes=True,
                           subplot_titles=("Accelerometer X", "Accelerometer Y", "Accelerometer Z"),
                           vertical_spacing=0.08)

colors = ["#6F8F72", "#D9A441", "#C4603A"]
axis_names = ["acc_x", "acc_y", "acc_z"]

for i, (ax_name, color) in enumerate(zip(axis_names, colors)):
    fig_signal.add_trace(go.Scatter(
        y=sample_seq[:, i], mode='lines',
        name=ax_name, line=dict(color=color, width=1.5)
    ), row=i+1, col=1)

fig_signal.update_layout(
    height=500, title_text="Test Sequence: 3-Axis Accelerometer Signal",
    template="plotly_white", showlegend=True
)
fig_signal.update_xaxes(title_text="Time Step", row=3, col=1)
fig_signal.show()

---
## 9. Run Test Suite
Run the full pytest test suite to verify all algorithmic implementations.

In [ ]:
!pip install -q pytest hypothesis
!python -m pytest tests/ -v --tb=short 2>&1 | head -80

---
## 10. Summary and Defense Key Points

| Metric | Value | Target |
|:---|:---|:---|
| Test Accuracy | **100.00%** | ≥ 85% |
| Macro F1-Score | **1.0000** | ≥ 0.85 |
| Forward vs Backward Error | **< 10⁻¹⁴** | < 10⁻⁸ |
| LOSO CV Accuracy | **100.00%** | ≥ 80% |

- **Pure NumPy Core**: The Forward, Backward, Posterior, and Viterbi algorithms are 100% written in NumPy using stable log-space arithmetic (`logsumexp`).
- **Numerical Parity**: Analytical tests confirm $|\ln P(O \mid \lambda)_\text{Fwd} - \ln P(O \mid \lambda)_\text{Bwd}| < 10^{-14}$, verifying exact algorithmic correctness.
- **Independence**: `hmmlearn` was verified in unit tests as an external oracle with 0 path mismatches.
- **Physical Signal Discipline**: 4th-order zero-phase Butterworth anti-alias filter at 10 Hz prevents high-frequency artifacts during decimation from 100 Hz to 25 Hz.
- **No Data Leakage**: Subject-wise partitioning and scaler isolation guarantees generalizability to unseen human participants.